## Housing in Mexico 

The Housing in Mexico dataset contains information about residential properties across different cities and states in Mexico. It includes features such as property price, location, property type, area size, number of bedrooms and bathrooms, parking spaces, and other amenities. 

This dataset is commonly used for data analysis and machine learning tasks, especially for predicting house prices and studying real-estate market trends. It is also useful for practicing data cleaning, visualization, and exploratory data analysis techniques.

# Project Workflow
### In this project I will follow the following Data Science and machine learning Workflow
***1- Prepare Data***    
    - Import  
    - Explore  
    - Split  

***2- Build Model***    
    - Baseline  
    - Iterate  
    - Evaluation  

***3- Communicating Results***   

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly_express as px 
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [12]:
# Import data raw file
# Reading the first raw data file
df1 = pd.read_csv("../data/raw/mexico-city-real-estate-1.csv")
df1.head()

,operation,property_type,place_with_parent_names,lat-lon,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url
0,sell,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",5500000.0,MXN,5450245.50,289775.66,NaN,54.0,NaN,101851.851852,NaN,NaN,NaN,http://miguel-hidalgo-df.properati.com.mx/o3zb...
1,sell,house,|Iztapalapa|Distrito Federal|México|,"19.31033,-99.068557",1512000.0,MXN,1498321.97,79661.96,NaN,80.0,NaN,18900.000000,NaN,NaN,NaN,http://iztapalapa.properati.com.mx/q7t0_venta_...
2,sell,apartment,|Tlalpan|Distrito Federal|México|,"19.279771,-99.234597",926667.0,MXN,918284.00,48822.82,NaN,100.0,NaN,9266.670000,NaN,NaN,NaN,http://tlalpan.properati.com.mx/qbi4_venta_dep...
3,sell,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",6410000.0,MXN,6352013.39,337720.36,NaN,135.0,NaN,47481.481481,NaN,NaN,NaN,http://miguel-hidalgo-df.properati.com.mx/opeq...
4,sell,apartment,|Benito Juárez|Quintana Roo|México|,"21.1902642,-86.8198375",875000.0,USD,16457437.50,875000.00,0.0,263.0,NaN,3326.996198,NaN,NaN,NaN,http://cancun.properati.com.mx/hg4t_venta_depa...


In [ ]:
# lat-lon column
# Creating a seperate lat and loan columns from the column "lat-lon"
df1[["lat", "lon"]]=df1["lat-lon"].str.split(",", expand=True)

In [ ]:
# drop the old "lat-lon column"
df1.drop(columns="lat-lon", inplace=True)

In [ ]:
df1.info()

In [ ]:
# cast the lat and lon coluymns to float
df1[["lat", "lon"]]= df1[["lat","lon"]].astype("float")

In [ ]:
df1.info()

In [ ]:
df1.head()

In [ ]:
# Extract the state column from "place with parents name column"
df1["state"]=df1["place_with_parent_names"].str.split("|", expand=True)[2]

In [ ]:
# Drop the "place_with_parent_name" column
df1.drop(["place_with_parent_names"], axis=1, inplace=True)

In [ ]:
df1.info()

In [ ]:
df1.head()

### Extracting new columns"price_usd" as in our dataset there is several columns decribing the price
### as the rate of USD to mexican pseo is 1 to 17 we can devide the price in peso "price" by 17 and get the price in USD

In [ ]:
df1["price_usd"] = df1["price"].apply(lambda x: x/17).astype("float")

In [ ]:
df1.head()

In [ ]:
df1.info()

In [ ]:
# According to the info the column "surface covered in m2" represents the area, let's rename it to be "area_m2"
df1.rename(columns={"surface_covered_in_m2": "area_m2"}, inplace=True)

In [ ]:
df1.head()

In [ ]:
# drop the columns that have large numbers of nulls
df1.drop(["rooms","floor","expenses"], axis=1, inplace=True)

In [ ]:
df1.info()

# Leakage
Now let's frop all the leaky columns that can harm out model performance.
leaky columns such as "surface_total_in_m2 ", "price_usd_per_m2"  as we have already ["area_m2", "price_usd"]
columns and such leakage could affect our model performance

In [ ]:
# drop leaky columns
df1.drop(columns=["price_usd_per_m2", "surface_total_in_m2"], inplace=True)

In [ ]:
df1.head()

# Multicollinearity
This happen when we have two or more features that have strong corelation to each other which is not good for our model.
We can discover it using corelation matrix

In [ ]:
df1.select_dtypes("float").corr()

In [ ]:
# Plotting a heat map to take a better look
sns.heatmap(df1.select_dtypes("float").corr())

In [ ]:
# It's clear the the features related to the price have a multicollinearity
# Drop the high colrelated to each other columns
df1.drop(columns=["price_aprox_local_currency","price_aprox_usd", "price_per_m2","price"], inplace=True)

In [ ]:
df1.head()

# Low and High Cardinality 
Low and high cardinality features acutally are useless to our model, high cardilality column such as property_url will never help our linear regression model to make good prediction as every observation has it's own value.
Likewise if we will take a look at the low cardilaty column operation and currency.

In [ ]:
df1.select_dtypes("str").nunique()

In [ ]:
df1.drop(columns=["currency","properati_url","operation"], inplace=True)

In [ ]:
df1.head()

In [ ]:
df1.info()

In [ ]:
# Dealing with the Outliers
fig,ax = plt.subplots(figsize=(15,6))
ax.boxplot(df1["area_m2"].fillna(df1["area_m2"].mean()),vert=False)
ax.set_xlabel("Area[m2]")
ax.set_title("Distribution of home sizes");

In [ ]:
df1.info()

In [ ]:
df1.head()

# EDA

***Exploratory Data Analysis***

In [ ]:
# Distribtion of the price for the apartments
df1=df1[df1["property_type"]=="apartment"]
df1.head()

In [ ]:
# Distribtion of the price for the apartments
fig,ax = plt.subplots(figsize=(10,6))
ax.hist(df1["price_usd"]);
plt.ticklabel_format(style='plain', axis='x')
ax.set_xlabel("Price[USD]")
ax.set_ylabel("Distribution")
ax.set_title("Distribution of apartments prices");

We can see that most of the apartments price between 100_000 and 400_000 USD, and we still have ofcourse outliers which maybe depends on the location of the apartment.

In [ ]:
# Lets figure out if the state has a relationship to the price's mean
state_grouped = df1.groupby("state")["price_usd"].mean().sort_values()
state_grouped

In [ ]:
# However it's clearly that the capital has the highest prices, let's plot it
state_grouped.plot(kind="barh", color="black", alpha=0.7)
plt.title("Mean price distribution by state");

In [ ]:
# Visualizing the prices distribution
fig = px.scatter_map(
    df1,
    lat="lat",
    lon="lon",
    center={"lat": 19.4326, "lon": -99.1332}, # map will center in mexico
    width=600,
    height=600,
    color="price_usd",
    hover_data=["price_usd"],  # Display price when hovering mouse over house
)

fig.update_layout(map_style="open-street-map")


In [ ]:
# make another mask for the price to get the prices < 400_000 usd
df1 = df1[df1["price_usd"]<400_000]


In [ ]:
df1.head()

In [ ]:
# Focusing on the apartmens that located in the capital Distrito Federal
df1 = df1[df1["state"]=="Distrito Federal"]
df1.head()

In [ ]:
# Checking the corelation between the price and the area
corr = df1["price_usd"].corr(df1["area_m2"])
corr

In [ ]:
# Dealing with outliers of the area 
low, high = df1["area_m2"].quantile([0.1,0.9])
low,high

# Mask for the area 
df1 = df1[df1["area_m2"].between(low,high)]

In [ ]:
df1 = df1.reset_index(drop=True)

The corelation cooffcient is 0.26 which is not a strong indicator that the price is highly affected by the area, on the other hand the location is more corelated to the price.

In [ ]:
# Visualizing the corelation between the price and the area
plt.scatter(df1["price_usd"], df1["area_m2"])
plt.xlabel("Price [usd]")
plt.ylabel("Area [m2]")
plt.title("Price vs Area")

So far we've made several steps to clean and mask our dataset and also some EDA using our visualization tools and our pandas tools, But it's a best practice to include all this work inside one class or fuction and use it as we will repeat all this steps with the other datasets.

In [27]:
from src.data import DataHandler
data_handler = DataHandler()
df = data_handler.wrangle_data("../data/raw/mexico-city-real-estate-1.csv")
df.head()


,area_m2,lat,lon,price_usd
0,54.0,23.634501,-102.552788,323529.411765
1,100.0,19.279771,-99.234597,54509.823529
2,135.0,23.634501,-102.552788,377058.823529
3,87.0,19.432657,-99.177444,259764.705882
4,100.0,19.367025,-99.170349,185294.117647


In [28]:
# Extract df from multiple raw files
df = data_handler.df_from_multiple_files("../data/raw/mexico-city-real-estate-*.csv")
df.head()

,area_m2,lat,lon,price_usd
0,54.0,23.634501,-102.552788,323529.411765
1,100.0,19.279771,-99.234597,54509.823529
2,135.0,23.634501,-102.552788,377058.823529
3,87.0,19.432657,-99.177444,259764.705882
4,100.0,19.367025,-99.170349,185294.117647


In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5848 entries, 0 to 5847
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   area_m2    5848 non-null   float64
 1   lat        5445 non-null   float64
 2   lon        5445 non-null   float64
 3   price_usd  5848 non-null   float64
dtypes: float64(4)
memory usage: 182.9 KB


In [30]:
# Save the clean data
data_handler.save_data(df=df, path="../data/processed/clean_data.csv")

# split the data

In [31]:
from sklearn.model_selection import train_test_split

In [32]:
df.head()

,area_m2,lat,lon,price_usd
0,54.0,23.634501,-102.552788,323529.411765
1,100.0,19.279771,-99.234597,54509.823529
2,135.0,23.634501,-102.552788,377058.823529
3,87.0,19.432657,-99.177444,259764.705882
4,100.0,19.367025,-99.170349,185294.117647


In [33]:
df.describe()

,area_m2,lat,lon,price_usd
count,5848.000000,5445.000000,5445.000000,5848.000000
mean,79.523598,19.477801,-99.219612,118156.246429
std,22.665505,0.591024,0.515818,84585.137725
min,50.000000,19.194247,-102.552788,8735.294118
25%,62.000000,19.365624,-99.184370,52411.764706
50%,74.000000,19.393650,-99.159469,88235.294118
75%,90.000000,19.437285,-99.137868,164705.882353
max,160.000000,23.634501,-90.488467,398823.529412


In [34]:
# vertical split
X = df.drop(["price_usd"], axis=1)
y=df["price_usd"]


In [35]:
# Horizontal split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.1, shuffle=True)

In [36]:
X_train.shape, y_train.shape

((5263, 3), (5263,))

In [37]:
X_test.shape, y_test.shape

((585, 3), (585,))

In [58]:
# Importing the linear regression model and the imputer
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

In [59]:
# dump model - baseline
y_pred_baseline= [y_train.mean() for i in y_train]
mae_baseline = round(mean_absolute_error(y_pred_baseline, y_train),2)
mae_baseline

67593.73

Here we got the mae_baseline 67593.73 and we need to beat this number when cauculating the mean absolute error for the training set


In [60]:
model = make_pipeline(
    
    SimpleImputer(),
    LinearRegression()
)

In [61]:
# fit the model
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['area_m2','lat','lon']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf

In [62]:
# y_pred_train
y_pred_train = model.predict(X_train)

In [63]:
y_pred_train

array([ 72890.88634201, 140715.27247411, 109165.4728865 , ...,
        83813.28703092,  64413.69540828, 196271.31027115], shape=(5263,))

In [64]:
# Evaluating the model performance on the training set
train_MAE = round(mean_absolute_error(y_train, y_pred_train),2)
train_MAE

52560.3

In [65]:
y_pred_test = model.predict(X_test)
test_MAE = round(mean_absolute_error(y_test, y_pred_test),2)
test_MAE

55310.05

In [57]:
from sklearn.ensemble import RandomForestRegressor
rfr = make_pipeline(SimpleImputer(), RandomForestRegressor())
rfr.fit(X_train, y_train)
round(mean_absolute_error(y_test, rfr.predict(X_test)),2)

26150.07

In [66]:
from xgboost import XGBRegressor
xgb = make_pipeline(SimpleImputer(), XGBRegressor())
xgb.fit(X_train, y_train)
round(mean_absolute_error(y_test, xgb.predict(X_test)),2)

27462.42

Model Selection

Based on the evaluation results, the Random Forest Regressor achieved the lowest Mean Absolute Error (MAE) among the models tested.

Since a lower MAE indicates that the model's predictions are, on average, closer to the actual apartment prices, the Random Forest Regressor was selected as the final model for deployment.



In [67]:
import pickle

# Saving the best model
with open("../models/rf_model.pkl", mode="wb") as f:
    pickle.dump(rfr,f)

# Communicating results

In [ ]:
# Extract the feature importances and the intercept
importances = rfr.feature_importances_
importances

In [ ]:
feature_names = X_train.columns
feature_names

In [ ]:
# Feature importances series
fea_imp = pd.Series(importances, index = feature_names)
fea_imp

In [ ]:
fea_imp.sort_values().plot(kind="barh")
plt.xlabel("Feature's weight")
plt.ylabel("Feature")
plt.title("Feature importances for aparment price");

In [68]:
from src.model import RFModel
rf_model = RFModel(path="../models/rf_model.pkl")


In [69]:
rf_model.make_prediction(area=134.16, lat=21.43, lon=-100.71)

np.float64(265566.94)

In [74]:
import requests

response = requests.post(url="http://0.0.0.0:8000/predict", json={"area":87.0, "lat": 19.432657, "lon": -99.177444})
print(response.status_code)
response.json()

200


{'success': True, 'prediction': 266173.65, 'message': 'Prediction done'}

# Prediction Function
one of the most important steps is to deploy the model and make it usable for the world even with a simple prediction function


## 🏡 Mexico Housing Price Prediction — Live ML App

Experience the deployed machine learning model built to estimate housing prices in Mexico.

🔗 **Live Demo:** https://housinginmexicomodel.streamlit.app/

### ✨ Features
- Real-time house price predictions
- User-friendly interactive interface
- Powered by a trained regression model
- Deployed using Streamlit Cloud

> Try different inputs and see how property features influence price predictions.

In [3]:
data = {
        "area": 60,
        "lat": 19.9,
        "lon": -99.16
    }
print(**data)

TypeError: print() got an unexpected keyword argument 'area'